# Palm Vein — Metric Learning with ResNet-18 + Triplet Loss
**Pipeline:** ResNet-18 backbone (ImageNet pretrained, last block unfrozen) → Dropout → Linear(512, 128) → L2-normalise

**Training:** PK-sampler batches + online batch-hard triplet mining

**Evaluation:** Leave-one-out nearest-neighbour accuracy + 5-fold cross-validation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os, random, math
from collections import defaultdict
from sklearn.model_selection import KFold

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device('cpu')
print('PyTorch:', torch.__version__)
print('Device :', device)

## Configuration
Tune these values in one place.

In [ ]:
DATASET      = r'c:\Users\riyat\OneDrive\Documents\palmvein\palm_vein_ready'

# Model
EMBEDDING_DIM = 128      # output embedding size
DROPOUT       = 0.5

# PK sampler
P = 15                   # subjects per batch
K = 4                    # images per subject per batch  →  batch = P*K = 60
BATCHES_PER_EPOCH = 20   # how many PK batches == 1 epoch

# Training
EPOCHS  = 50
LR      = 1e-4
MARGIN  = 0.2            # triplet loss margin (unit-normalised embeddings, range [0,2])

# K-fold
N_FOLDS = 5

# ImageNet stats (required because backbone was trained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f'Batch size : {P*K}  ({P} subjects x {K} images)')
print(f'Embedding  : {EMBEDDING_DIM}-d, L2-normalised (unit hypersphere)')

## Transforms
Augmentation is applied **before** normalisation so it operates on realistic pixel ranges.

- **Rotation ±180°** — handles capture angle variation
- **RandomResizedCrop** — handles distance/scale variation (85%–115%)
- **ColorJitter** — slight brightness variation
- **Normalise** — ImageNet mean/std (required for frozen pretrained layers)

In [ ]:
train_transform = T.Compose([
    T.Grayscale(num_output_channels=3),                        # grayscale → 3-ch for ResNet
    T.RandomRotation(degrees=180, fill=0),                     # rotation augmentation
    T.RandomResizedCrop(224, scale=(0.85, 1.15),
                        ratio=(0.95, 1.05)),                   # scale/distance augmentation
    T.ColorJitter(brightness=0.1),                             # brightness jitter
    T.ToTensor(),                                              # [0,255] → [0,1]
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),         # ImageNet normalisation
])

val_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),         # no augmentation at val/test
])

# Quick sanity check
sample = Image.open(os.path.join(DATASET, 'subject001', 'sample_000.png'))
t = train_transform(sample)
print('Train transform output shape:', t.shape)    # expect [3, 224, 224]
print('Value range: [{:.3f}, {:.3f}]'.format(t.min().item(), t.max().item()))

## Dataset
Loads every image from palm_vein_ready/. Each subject gets an integer label 0–59.

In [ ]:
class PalmVeinDataset(Dataset):
    def __init__(self, root, subjects, transform=None):
        self.transform = transform
        self.samples   = []          # list of (path, label_int)
        self.label_to_indices = defaultdict(list)
        for label, subj in enumerate(subjects):
            folder = os.path.join(root, subj)
            for fname in sorted(os.listdir(folder)):
                if fname.endswith('.png'):
                    idx = len(self.samples)
                    self.samples.append((os.path.join(folder, fname), label))
                    self.label_to_indices[label].append(idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path)
        if self.transform:
            img = self.transform(img)
        return img, label


# Load all 60 subjects
all_subjects = sorted([d for d in os.listdir(DATASET)
                        if os.path.isdir(os.path.join(DATASET, d))])
print(f'Subjects: {len(all_subjects)},  Images: {len(all_subjects)*5}')

## PK Sampler
Each iteration samples **P** subjects and **K** images per subject → guaranteed positives in every batch.

In [ ]:
class PKSampler(Sampler):
    """
    Samples P classes and K images per class per batch.
    Yields indices for one epoch = BATCHES_PER_EPOCH batches.
    """
    def __init__(self, label_to_indices, P, K, batches_per_epoch):
        self.label_to_indices  = label_to_indices
        self.P                 = P
        self.K                 = K
        self.batches_per_epoch = batches_per_epoch
        self.labels            = list(label_to_indices.keys())

    def __iter__(self):
        for _ in range(self.batches_per_epoch):
            chosen_classes = random.sample(self.labels, self.P)
            batch_indices  = []
            for cls in chosen_classes:
                # sample K with replacement in case a class has fewer than K images
                idxs = random.choices(self.label_to_indices[cls], k=self.K)
                batch_indices.extend(idxs)
            yield from batch_indices

    def __len__(self):
        return self.batches_per_epoch * self.P * self.K

print('PKSampler defined. Batch size =', P*K)

## Embedding Network
ResNet-18 (ImageNet pretrained) → all layers **frozen** except layer4 and the new head.

Head: Flatten → Dropout → Linear(512, 128) → L2-normalise

BatchNorm layers kept in **eval mode** during training to avoid corrupting running stats with small batches.

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=EMBEDDING_DIM, dropout=DROPOUT):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Freeze everything first
        for param in backbone.parameters():
            param.requires_grad = False

        # Unfreeze only the last residual block (layer4)
        for param in backbone.layer4.parameters():
            param.requires_grad = True

        # Strip the classification head; keep up to & including avgpool
        # ResNet children: conv1, bn1, relu, maxpool, layer1-4, avgpool, fc
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # output: [B, 512, 1, 1]

        # New embedding head
        self.head = nn.Sequential(
            nn.Flatten(),                          # [B, 512]
            nn.Dropout(p=dropout),
            nn.Linear(512, embedding_dim),
        )

    def forward(self, x):
        x = self.backbone(x)       # [B, 512, 1, 1]
        x = self.head(x)           # [B, embedding_dim]
        x = F.normalize(x, p=2, dim=1)   # L2-normalise → unit hypersphere
        return x

    def train(self, mode=True):
        """Override train() to keep all BatchNorm layers in eval mode."""
        super().train(mode)
        for m in self.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                m.eval()
        return self


model = EmbeddingNet().to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,}  ({100*trainable/total:.1f}% of {total:,} total)')
print(f'Output shape     : {model(torch.zeros(1,3,224,224)).shape}')   # expect [1, 128]

## Batch-Hard Triplet Loss
For each anchor: find the **hardest positive** (same class, largest distance) and **hardest negative** (different class, smallest distance) within the batch.

L = mean( max(0, D(a,p)² − D(a,n)² + margin) )

In [ ]:
def batch_hard_triplet_loss(embeddings, labels, margin=MARGIN):
    """
    embeddings : [B, D] float tensor, L2-normalised
    labels     : [B]   int tensor
    Returns scalar loss.
    """
    B = embeddings.size(0)

    # Pairwise squared Euclidean distances  [B, B]
    dist = torch.cdist(embeddings, embeddings, p=2).pow(2)

    # Boolean masks
    labels_col = labels.unsqueeze(1)          # [B,1]
    pos_mask = (labels_col == labels_col.T)   # same class  [B,B]
    neg_mask = ~pos_mask                      # diff class  [B,B]

    # Remove diagonal from positive mask (self-distance = 0)
    pos_mask.fill_diagonal_(False)

    # Hardest positive: max distance per anchor among true positives
    # If a subject has only 1 image in the batch, pos distance is 0
    hardest_pos = (dist * pos_mask.float()).max(dim=1).values   # [B]

    # Hardest negative: min distance per anchor among negatives
    dist_neg = dist.clone()
    dist_neg[~neg_mask] = float('inf')        # mask out non-negatives
    hardest_neg = dist_neg.min(dim=1).values  # [B]

    # Triplet loss
    loss = F.relu(hardest_pos - hardest_neg + margin)
    return loss.mean()

print('Batch-hard triplet loss defined. Margin =', MARGIN)

## Training Loop

In [ ]:
def train_one_epoch(model, loader, optimiser):
    model.train()
    total_loss = 0.0
    n_batches  = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        optimiser.zero_grad()
        embeddings = model(images)
        loss = batch_hard_triplet_loss(embeddings, labels)
        loss.backward()
        optimiser.step()
        total_loss += loss.item()
        n_batches  += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate_nn(model, dataset):
    """
    Leave-one-out nearest-neighbour accuracy.
    For each image: embed it, find closest other embedding, check label match.
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    all_emb, all_lbl = [], []
    for imgs, lbls in loader:
        all_emb.append(model(imgs.to(device)).cpu())
        all_lbl.extend(lbls.tolist())
    emb = torch.cat(all_emb)             # [N, D]
    lbl = torch.tensor(all_lbl)          # [N]

    dist = torch.cdist(emb, emb, p=2)   # [N, N]
    dist.fill_diagonal_(float('inf'))    # exclude self
    preds = lbl[dist.argmin(dim=1)]      # nearest neighbour label
    acc   = (preds == lbl).float().mean().item()
    return acc

print('Training and evaluation functions defined.')

## Single Train / Val Split
Quick run to verify everything works before full k-fold.

**Split:** subjects 0–47 → train, subjects 48–59 → val (80/20 on subjects)

In [ ]:
train_subjects = all_subjects[:48]
val_subjects   = all_subjects[48:]

train_ds = PalmVeinDataset(DATASET, train_subjects, transform=train_transform)
val_ds   = PalmVeinDataset(DATASET, val_subjects,   transform=val_transform)

pk_sampler = PKSampler(train_ds.label_to_indices, P=P, K=K,
                        batches_per_epoch=BATCHES_PER_EPOCH)
train_loader = DataLoader(train_ds, batch_size=P*K, sampler=pk_sampler)

model     = EmbeddingNet().to(device)
optimiser = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

train_losses, val_accs = [], []

print(f'Train subjects: {len(train_subjects)}  ({len(train_ds)} images)')
print(f'Val   subjects: {len(val_subjects)}  ({len(val_ds)} images)')
print(f'Starting training for {EPOCHS} epochs...')
print('-' * 50)

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader, optimiser)
    acc  = evaluate_nn(model, val_ds)
    train_losses.append(loss)
    val_accs.append(acc)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  |  loss: {loss:.4f}  |  val NN-acc: {acc*100:.1f}%')

print('-' * 50)
print(f'Best val accuracy: {max(val_accs)*100:.1f}%  (epoch {val_accs.index(max(val_accs))+1})')

## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(train_losses, color='steelblue')
ax1.set_title('Triplet Loss (train)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot([a*100 for a in val_accs], color='darkorange')
ax2.set_title('NN Accuracy (val)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy %')
ax2.set_ylim(0, 105)
ax2.axhline(max(val_accs)*100, color='red', linestyle='--', alpha=0.5,
            label=f'Best: {max(val_accs)*100:.1f}%')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r'c:\Users\riyat\OneDrive\Documents\palmvein\training_curves.png', dpi=150)
plt.show()

## 5-Fold Cross-Validation
Splits the 60 subjects into 5 folds of 12. Each fold trains fresh from pretrained weights.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
subjects_arr = np.array(all_subjects)
fold_accs = []

for fold, (train_idx, val_idx) in enumerate(kf.split(subjects_arr), 1):
    t_subj = subjects_arr[train_idx].tolist()
    v_subj = subjects_arr[val_idx].tolist()

    t_ds = PalmVeinDataset(DATASET, t_subj, transform=train_transform)
    v_ds = PalmVeinDataset(DATASET, v_subj, transform=val_transform)

    sampler = PKSampler(t_ds.label_to_indices, P=min(P, len(t_subj)),
                         K=K, batches_per_epoch=BATCHES_PER_EPOCH)
    loader  = DataLoader(t_ds, batch_size=P*K, sampler=sampler)

    m   = EmbeddingNet().to(device)
    opt = torch.optim.Adam(
        filter(lambda p: p.requires_grad, m.parameters()), lr=LR)

    for epoch in range(EPOCHS):
        train_one_epoch(m, loader, opt)

    acc = evaluate_nn(m, v_ds)
    fold_accs.append(acc)
    print(f'Fold {fold}/{N_FOLDS}  |  val NN-acc: {acc*100:.1f}%')

print('='*40)
print(f'Mean accuracy : {np.mean(fold_accs)*100:.1f}%')
print(f'Std           : {np.std(fold_accs)*100:.1f}%')